In [30]:
import os, json
from langchain_core.messages import message_to_dict, messages_from_dict
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage


class FileChatMessageHistory(BaseChatMessageHistory):
    def __init__(self, session_id, storage_path):
        self.session_id = session_id
        self.storage_path = storage_path
        self.file_path = os.path.join(self.storage_path, f"{self.session_id}.json")
        os.makedirs(os.path.dirname(self.file_path),exist_ok=True)
    

    def add_messages(self, message):
        all_messages = list(self.messages)
        all_messages.extend(message)

        new_messages = []
        for message in all_messages:
            d = message_to_dict(message)
            new_messages.append(d)

        # new_messages = [message_to_dict(message) for message in all_messages]

        with open(self.file_path,"w",encoding="utf-8") as f:
            json.dump(new_messages,f,indent=2)

    @property
    def messages(self) -> list[BaseMessage]:
        try:
            with open(self.file_path,"r", encoding="utf-8") as f:
                messages_data = json.load(f)
                return messages_from_dict(messages_data)
        except FileNotFoundError:
            return []
        
    def clear(self) -> None:
        with open(self.file_path,"w", encoding="utf-8") as f:
            json.dump([],f)

 


In [31]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder


In [32]:
deepseek = ChatOllama(
    model= "deepseek-r1:8b"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","u are helpful assistant."),
        MessagesPlaceholder("history_messages"),
        ("human","{input}"),
    ]
)

def print_prompt(prompt):
    print("\n","*"*30,"\n",prompt.to_string())
    return prompt

str_parser = StrOutputParser()

base_chain = prompt | deepseek | str_parser


def get_history(session_id):

    return FileChatMessageHistory(session_id = session_id,storage_path="./chat_history")



conservation_chain = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history_messages"
)

session_config = {
    "configurable":{
        "session_id":"smndyy"
    }
}


In [33]:
response1 = conservation_chain.stream({"input":"I am a Hopkins student"},session_config)
print("I am a Hopkins student.\n")
for chunk in response1:
    print(chunk,end="",flush=True)

response2 = conservation_chain.stream({"input":"My major is Data Science"},session_config)
print("\nMy major is Data Science.\n")
for chunk in response2:
    print(chunk,end="",flush=True)

response3 = conservation_chain.stream({"input":"Can u give me some suggestions in terms of job hunting"},session_config)
print("\nCan u give me some suggestions in terms of job hunting?\n")
for chunk in response3:
    print(chunk,end="",flush=True)


I am a Hopkins student.

Nice to meet you! Johns Hopkins is a fantastic university with a lot to offer. Whether you're at Homewood, Krieger, Peabody, or one of their other schools, I'm here to help with any questions or topics you might have. Let me know how I can assist you!
My major is Data Science.

Ah, great choice! Data Science is an incredibly exciting and versatile field with huge potential. Johns Hopkins University has a strong reputation in areas related to Data Science, including computer science, statistics, and quantitative fields, which provides a great foundation.

Data Science involves a mix of:
*   **Programming:** Languages like Python and R are essential.
*   **Mathematics & Statistics:** Linear algebra, calculus, probability, and statistics are fundamental for understanding data and building models.
*   **Data Engineering:** Handling large datasets, data cleaning, and data pipelines.
*   **Machine Learning:** Building models that learn from data.
*   **Data Visualiza